# **Complete Data Processing Pipeline**

This notebook contains the comprehensive data processing pipeline that transforms the raw `SERVICE_ORDER_BASE.xlsx` dataset into a clean, ML-ready format.

## **Processing Steps**

The pipeline is structured into the following key stages:

1. **Load Dataset:** Read the initial `SERVICE_ORDER_BASE.xlsx` file into a pandas DataFrame.

2. **Remove Unnecessary Columns:** Drop columns that are irrelevant for analysis or modeling, such as descriptive fields or identifiers with high cardinality that will not be used.

3. **Format Columns and Dates:** Standardize column names (e.g., `COUNTER OF SERVICE ORDER` to `ODOMETER`) and convert date formats from `YYYYMMDD` to `DD/MM/YYYY` for easier parsing.

4. **Create Code-Name Mappings:** Generate and store mappings between codes and their corresponding human-readable names (e.g., `MODEL TYPE CODE` to `ASSET NAME`) before dropping the name columns to save memory.

5. **Simplify `TIER` Column:** Convert categorical `TIER` values (e.g., "TIER 1 - T1") into numerical representations (1, 2) for easier processing.

6. **Convert `ASSET STATUS` to Binary:** Map the `ASSET STATUS` column to binary values (`ACTIVE` -> 1, `INACTIVE` -> 0).

7. **Convert `PREVENTIVE_CORRECTIVE MAINTENANCE` to Binary:** Transform the `PREVENTIVE_CORRECTIVE MAINTENANCE` column into a binary format (`PREVENTIVE` -> 1, `CORRECTIVE` -> 0).

8. **Remove Incomplete Rows:** Eliminate rows with missing values in critical financial columns (`PRODUCT QUANTITY`, `UNIT VALUE`, `GRAND TOTAL`) to ensure data quality.

9. **Extract Temporal Features:** Decompose the `SERVICE ORDER ORIGINAL DATE` into year, month, and day to enable time-based analysis.

10. **Label Encode `MODEL TYPE CODE`:** Convert the categorical `MODEL TYPE CODE` into numerical labels to reduce dimensionality.

11. **Label Encode `PRODUCT CODE`:** Apply label encoding to the `PRODUCT CODE` for efficient numerical representation.

12. **Label Encode `ASSET CODE`:** Transform vehicle plates in `ASSET CODE` into numerical labels.

13. **Treat Outliers using IQR Method:** Identify and cap outliers in cost-related columns (`GRAND TOTAL`, `UNIT VALUE`, `PRODUCT QUANTITY`) using the Interquartile Range (IQR) method to mitigate their impact on model performance.

14. **Reorder Columns:** Arrange the columns in a logical order (Vehicle -> Model -> Service -> Date -> Product -> Financial) to match the original dataset's structure.

### **Initial configuring**

Import necessary libraries and define file paths and global variables for the pipeline.

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import LabelEncoder
from datetime import datetime

# File paths
input_file = "data/SERVICE_ORDER_BASE.xlsx"
output_file = "data/SERVICE_ORDER_BASE_clean.xlsx"
mappings_file = "data/code_name_mappings.json"

# Store original values for encoding
original_model_types = None
original_product_codes = None
original_asset_codes = None

all_mappings = {}

# Define code-name pairs
code_name_pairs = [
  ("MODEL TYPE CODE", "ASSET NAME"),
  ("ASSET FAMILY CODE", "FAMILY NAME"),
  ("MANUFACTURER CODE", "MANUFACTURER NAME"),
  ("PRODUCT CODE", "PRODUCT DESCRIPTION")
]

columns_to_drop = []

## 1. Loading dataset

The raw dataset is loaded from an Excel file into a pandas DataFrame. We print the initial shape and column count to verify that the data has been loaded correctly.

In [ ]:
df = pd.read_excel(input_file)
print(f"Dataset loaded: {df.shape}")
print(f"Original columns: {len(df.columns)}")

## 2. Removing unnecessary columns

Columns that do not provide valuable information for predictive modeling are removed. This step simplifies the dataset and reduces memory usage. Examples include descriptive text fields (MODEL TYPE DESCRIPTION), redundant identifiers (SERVICE ORDER), and supplier details.

In [ ]:
columns_to_remove = [
  "MODEL TYPE DESCRIPTION",
  "ASSET PURCHASE DATE",
  "ITEM OF LEDGER ACCOUNT",
  "LEDGER ACCOUNT DESCRIPTION",
  "MAINTENANCE TYPE",
  "SERVICE ORDER",
  "INVOICE",
  "SUPPLIER'S CODE",
  "SUPPLIER'S STORE",
  "NAME OR COMPANY NAME"
]

existing_columns_to_remove = [col for col in columns_to_remove if col in df.columns]
df = df.drop(columns=existing_columns_to_remove)
print(f"Removed {len(existing_columns_to_remove)} columns")

## 3. Formating columns and dates

This section standardizes the dataset for consistency.

1. The `COUNTER OF SERVICE ORDER` column is renamed to `ODOMETER` for clarity.

2. The `SERVICE ORDER ORIGINAL DATE` is converted from a `YYYYMMDD` integer format to a more conventional `DD/MM/YYYY` string format, which facilitates date parsing in subsequent steps.

In [ ]:
# Rename COUNTER column
if "COUNTER  OF SERVICE ORDER" in df.columns:
  df = df.rename(columns={"COUNTER  OF SERVICE ORDER": "ODOMETER"})
  print("Renamed 'COUNTER  OF SERVICE ORDER' to 'ODOMETER'")

# Format dates from YYYYMMDD to DD/MM/YYYY
if "SERVICE ORDER ORIGINAL DATE" in df.columns:
  def format_date(date_str):
    if pd.isna(date_str):
      return date_str
    try:
      date_str = str(int(date_str))
      if len(date_str) == 8:
        year = date_str[:4]
        month = date_str[4:6]
        day = date_str[6:8]
        return f"{day}/{month}/{year}"
      return date_str
    except:
      return date_str

  df["SERVICE ORDER ORIGINAL DATE"] = df["SERVICE ORDER ORIGINAL DATE"].apply(format_date)
  print("Formatted dates from YYYYMMDD to DD/MM/YYYY")

## 4. Creating code-name mappings

To prepare the data for machine learning, categorical text columns are dropped in favor of their numerical code equivalents. Before dropping them, we create and store a JSON mapping file. This file allows for easy decoding of predictions or analysis results back into a human-readable format.

In [ ]:
for code_col, name_col in code_name_pairs:
  if code_col in df.columns and name_col in df.columns:
    # Create mapping dictionary
    mapping = dict(zip(df[code_col], df[name_col]))
    mapping_key = f"{code_col}_to_{name_col.replace(' ', '_')}"
    all_mappings[mapping_key] = mapping
    columns_to_drop.append(name_col)
    print(f"Created {code_col}_to_{name_col.replace(' ', '_')} with {len(mapping)} mappings")

# Add additional mappings for converted values
all_mappings["ASSET_STATUS_to_CODE"] = {"0": "INACTIVE", "1": "ACTIVE"}
all_mappings["TIER_to_CODE"] = {"1": "TIER 1", "2": "TIER 2"}
all_mappings["PREVENTIVE_CORRECTIVE_MAINTENANCE_to_CODE"] = {"0": "CORRECTIVE", "1": "PREVENTIVE"}

# Drop name columns
df = df.drop(columns=columns_to_drop)
print(f"Removed {len(columns_to_drop)} name columns (kept codes)")

## 5. Simplify TIER column

The `TIER` column contains string identifiers like **"TIER 1 - T1"**. These are mapped to simple integers (1, 2) to convert them into a numerical format suitable for modeling.

In [ ]:
if "TIER" in df.columns:
  # Map TIER values to numbers
  tier_mapping = {
    "TIER 1 - T1": 1,
    "TIER 2 - T2": 2
  }

  df["TIER"] = df["TIER"].map(tier_mapping)
  tier_counts = df["TIER"].value_counts().to_dict()
  print(f"TIER simplified: {tier_counts}")

## 6. Converting ASSET STATUS to binary

The `ASSET STATUS` column, with values **"ACTIVE"** and **"INACTIVE"**, is converted into a binary representation (1 and 0). This is a standard preprocessing step for categorical features with two distinct values.

In [ ]:
if "ASSET STATUS" in df.columns:
  status_mapping = {
    "ACTIVE": 1,
    "INACTIVE": 0
  }

  df["ASSET STATUS"] = df["ASSET STATUS"].map(status_mapping)
  status_counts = df["ASSET STATUS"].value_counts().to_dict()
  print(f"ASSET STATUS converted: {status_counts}")

## **7. Converting PREVENTIVE_CORRECTIVE MAINTENANCE to binary**

Similarly, the `PREVENTIVE_CORRECTIVE MAINTENANCE` column is mapped to binary values (`PREVENTIVE` -> 1, `CORRECTIVE` -> 0), making it ready for numerical analysis.

In [ ]:
if "PREVENTIVE_CORRECTIVE MAINTENANCE" in df.columns:
  maintenance_mapping = {
    "PREVENTIVE": 1,
    "CORRECTIVE": 0
  }

  df["PREVENTIVE_CORRECTIVE MAINTENANCE"] = df["PREVENTIVE_CORRECTIVE MAINTENANCE"].map(maintenance_mapping)
  maintenance_counts = df["PREVENTIVE_CORRECTIVE MAINTENANCE"].value_counts().to_dict()
  print(f"MAINTENANCE converted: {maintenance_counts}")

## 8. Removing incomplete rows

Rows with missing values in key financial columns (`PRODUCT QUANTITY`, `UNIT VALUE`, `GRAND TOTAL`) are dropped. This ensures the integrity of the dataset for any financial analysis or modeling task. We also calculate and print the data retention rate to monitor how much data was lost in this step.

In [ ]:
required_columns = ["PRODUCT QUANTITY", "UNIT VALUE", "GRAND TOTAL"]

initial_count = len(df)
df = df.dropna(subset=required_columns)
final_count = len(df)
removed_count = initial_count - final_count
retention_rate = (final_count / initial_count) * 100

print(f"Removed {removed_count} incomplete rows")
print(f"Data retention: {retention_rate:.2f}%")

## **9. Extracting temporal features from SERVICE ORDER ORIGINAL DATE**

Temporal information is crucial for time-series analysis and identifying trends. We extract the `year`, `month`, and `day` from the `SERVICE ORDER ORIGINAL DATE` column. This creates three new numerical features. The original date column is then dropped, and any rows with invalid date formats are removed.

In [ ]:
if "SERVICE ORDER ORIGINAL DATE" in df.columns:
  def extract_temporal_features(date_str):
    if pd.isna(date_str):
      return None, None, None
    try:
      # Parse DD/MM/YYYY format
      day, month, year = date_str.split('/')
      return int(year), int(month), int(day)
    except:
      return None, None, None

  # Extract features
  temporal_data = df["SERVICE ORDER ORIGINAL DATE"].apply(extract_temporal_features)
  df["SERVICE_ORDER_year"] = [x[0] for x in temporal_data]
  df["SERVICE_ORDER_month"] = [x[1] for x in temporal_data]
  df["SERVICE_ORDER_day"] = [x[2] for x in temporal_data]

  # Remove original date column
  df = df.drop(columns=["SERVICE ORDER ORIGINAL DATE"])

  # Remove rows with invalid dates
  df = df.dropna(subset=["SERVICE_ORDER_year", "SERVICE_ORDER_month", "SERVICE_ORDER_day"])

  print("Extracted 3 temporal features")
  print("Features: year, month, day")
  print(f"Date range: {df['SERVICE_ORDER_year'].min():.0f}-01-01 to {df['SERVICE_ORDER_year'].max():.0f}-08-05")
  print(f"New dataset shape: {df.shape}")

## **10. Label encoding MODEL TYPE CODE**

The `MODEL TYPE CODE` is a high-cardinality categorical feature. One-hot encoding would create a very large number of new columns, leading to inefficiency. Instead, we use label encoding to convert each unique model code into a unique integer. A mapping from the encoded value back to the original is saved for future reference.

In [ ]:
if "MODEL TYPE CODE" in df.columns:
  original_model_types = df["MODEL TYPE CODE"].unique()
  unique_model_types = len(original_model_types)

  print(f"MODEL TYPE CODE has {unique_model_types} unique values")
  print(f"Using label encoding (much more efficient than {unique_model_types} one-hot columns)")

  # Label encoding
  label_encoder = LabelEncoder()
  df["MODEL_TYPE_CODE_encoded"] = label_encoder.fit_transform(df["MODEL TYPE CODE"])

  # Create mapping for decoding
  model_type_mapping = dict(zip(label_encoder.transform(original_model_types), original_model_types))
  all_mappings["MODEL_TYPE_CODE_encoded_to_original"] = model_type_mapping

  # Remove original column
  df = df.drop(columns=["MODEL TYPE CODE"])

  print("Created 1 label-encoded column (MODEL_TYPE_CODE_encoded)")
  print("Removed original MODEL TYPE CODE column")
  print(f"Mapped {unique_model_types} values to integers 0-{unique_model_types-1}")
  print(f"New dataset shape: {df.shape}")
else:
  print("MODEL TYPE CODE column not found")

## 11. Label encoding PRODUCT CODE

Following the same logic as with `MODEL TYPE CODE`, the `PRODUCT CODE` column is label-encoded. This efficiently converts the large number of unique product codes into a single numerical feature.

In [ ]:
if "PRODUCT CODE" in df.columns:
  original_product_codes = df["PRODUCT CODE"].unique()
  unique_product_codes = len(original_product_codes)

  print(f"PRODUCT CODE has {unique_product_codes} unique values")
  print(f"Using label encoding (much more efficient than {unique_product_codes} one-hot columns)")

  # Label encoding
  label_encoder = LabelEncoder()
  df["PRODUCT_CODE_encoded"] = label_encoder.fit_transform(df["PRODUCT CODE"])

  # Create mapping for decoding
  product_code_mapping = dict(zip(label_encoder.transform(original_product_codes), original_product_codes))
  all_mappings["PRODUCT_CODE_encoded_to_original"] = product_code_mapping

  # Remove original column
  df = df.drop(columns=["PRODUCT CODE"])

  print("Created 1 label-encoded column (PRODUCT_CODE_encoded)")
  print("Removed original PRODUCT CODE column")
  print(f"Mapped {unique_product_codes} values to integers 0-{unique_product_codes-1}")
  print(f"New dataset shape: {df.shape}")
else:
  print("PRODUCT CODE column not found")

## 12. Label encoding ASSET CODE (vehicle plates)

The `ASSET CODE`, which represents unique vehicle plates, is also a high-cardinality feature. Label encoding is applied to convert these identifiers into a numerical format, which is essential for algorithms that require numerical input.

In [ ]:
if "ASSET CODE" in df.columns:
  original_asset_codes = df["ASSET CODE"].unique()
  unique_asset_codes = len(original_asset_codes)

  print(f"ASSET CODE has {unique_asset_codes} unique vehicle plates")
  print(f"Using label encoding (much more efficient than {unique_asset_codes} one-hot columns)")

  # Label encoding
  label_encoder = LabelEncoder()
  df["ASSET_CODE_encoded"] = label_encoder.fit_transform(df["ASSET CODE"])

  # Create mapping for decoding
  asset_code_mapping = dict(zip(label_encoder.transform(original_asset_codes), original_asset_codes))
  all_mappings["ASSET_CODE_encoded_to_original"] = asset_code_mapping

  # Remove original column
  df = df.drop(columns=["ASSET CODE"])

  print("Created 1 label-encoded column (ASSET_CODE_encoded)")
  print("Removed original ASSET CODE column")
  print(f"Mapped {unique_asset_codes} vehicle plates to integers 0-{unique_asset_codes-1}")
  print(f"New dataset shape: {df.shape}")
else:
  print("ASSET CODE column not found")

## 13. Treating outliers using IQR method

Outliers in cost-related columns can skew statistical analyses and degrade the performance of machine learning models. We apply the Interquartile Range (IQR) method to identify and treat these outliers. Values falling outside `1.5 * IQR` from the first and third quartiles are "capped," meaning they are replaced with the calculated lower or upper bound. This approach mitigates the effect of extreme values without removing the entire row.

In [ ]:
# Define columns to treat for outliers
cost_columns = ['GRAND TOTAL', 'UNIT VALUE', 'PRODUCT QUANTITY']
outlier_summary = {}

for column in cost_columns:
  if column in df.columns:
    original_count = len(df)
    data = df[column].dropna()

    # Calculate IQR bounds
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers before treatment
    outliers_before = len(data[(data < lower_bound) | (data > upper_bound)])
    outlier_percentage = outliers_before / len(data) * 100

    # Cap outliers to bounds
    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)

    outlier_summary[column] = {
      'outliers_treated': outliers_before,
      'percentage': outlier_percentage,
      'lower_bound': lower_bound,
      'upper_bound': upper_bound
    }

    print(f"{column}: {outliers_before} outliers ({outlier_percentage:.2f}%) capped to bounds [{lower_bound:.2f}, {upper_bound:.2f}]")

print(f"Outlier treatment completed for {len(cost_columns)} columns")
print(f"Dataset shape after outlier treatment: {df.shape}")

## **14. Defining the Primary Column Order**

First, an empty list called `desired_order` is created. The code then checks if specific columns exist in the DataFrame. If they do, they are added to the list in a predefined, logical sequence, grouped by category (asset identification, classification, service information, etc.).

In [ ]:
# Define the desired column order (similar to original dataset)
desired_order = []

# Vehicle/Asset identification
if "ASSET_CODE_encoded" in df.columns:
  desired_order.append("ASSET_CODE_encoded")

# Model and classification
if "MODEL_TYPE_CODE_encoded" in df.columns:
  desired_order.append("MODEL_TYPE_CODE_encoded")
if "ASSET_FAMILY_CODE_encoded" in df.columns:
  desired_order.append("ASSET_FAMILY_CODE_encoded")
if "MANUFACTURER_CODE_encoded" in df.columns:
  desired_order.append("MANUFACTURER_CODE_encoded")
if "MANUFACTURE_YEAR_encoded" in df.columns:
  desired_order.append("MANUFACTURE_YEAR_encoded")
if "ASSET STATUS" in df.columns:
  desired_order.append("ASSET STATUS")
if "TIER" in df.columns:
  desired_order.append("TIER")

# Service/maintenance information
if "ODOMETER" in df.columns:
  desired_order.append("ODOMETER")

# Date information
if "SERVICE_ORDER_year" in df.columns:
  desired_order.append("SERVICE_ORDER_year")
if "SERVICE_ORDER_month" in df.columns:
  desired_order.append("SERVICE_ORDER_month")
if "SERVICE_ORDER_day" in df.columns:
  desired_order.append("SERVICE_ORDER_day")

# Product and service details
if "PRODUCT_CODE_encoded" in df.columns:
  desired_order.append("PRODUCT_CODE_encoded")
if "PRODUCT QUANTITY" in df.columns:
  desired_order.append("PRODUCT QUANTITY")
if "UNIT VALUE" in df.columns:
  desired_order.append("UNIT VALUE")
if "GRAND TOTAL" in df.columns:
  desired_order.append("GRAND TOTAL")
if "PREVENTIVE_CORRECTIVE MAINTENANCE" in df.columns:
  desired_order.append("PREVENTIVE_CORRECTIVE MAINTENANCE")

## **15. Finalizing Column Order and Reordering the DataFrame**

This block ensures that no columns are lost. It identifies any columns that exist in the DataFrame but were not added to the desired_order list and appends them to the end. Finally, the DataFrame df is reordered according to this completed list.

In [ ]:
# Add any remaining columns that weren't explicitly ordered
remaining_columns = [col for col in df.columns if col not in desired_order]
desired_order.extend(remaining_columns)

# Reorder the dataframe
df = df[desired_order]

print(f"Reordered {len(df.columns)} columns to match original structure")
print(f"Order: Vehicle -> Model -> Service -> Date -> Product -> Financial")
print(f"Final dataset shape: {df.shape}")

## **16. Preparing and Saving Code Mappings**

To save the mappings to a JSON file, a helper function (convert_numpy_types) is required to convert NumPy-specific data types (like np.integer) into native Python types (like int), which are JSON serializable. After conversion, the mappings dictionary is saved to the mappings_file.

In [ ]:
# Save mappings with numpy type conversion
print("\nSaving code-name mappings...")

def convert_numpy_types(obj):
  if hasattr(obj, 'tolist'):
    return obj.tolist()
  elif isinstance(obj, np.integer):
    return int(obj)
  elif isinstance(obj, np.floating):
    return float(obj)
  elif isinstance(obj, dict):
    return {convert_numpy_types(key): convert_numpy_types(value) for key, value in obj.items()}
  elif isinstance(obj, list):
    return [convert_numpy_types(item) for item in obj]
  elif isinstance(obj, tuple):
    return tuple(convert_numpy_types(item) for item in obj)
  else:
    return obj

all_mappings_serializable = convert_numpy_types(all_mappings)

with open(mappings_file, 'w', encoding='utf-8') as f:
  json.dump(all_mappings_serializable, f, indent=2, ensure_ascii=False)
print(f"Saved {len(all_mappings)} mapping dictionaries to {mappings_file}")

## **17. Saving the Final Dataset**
The processed and reordered DataFrame is saved to a new Excel file (output_file), without including the pandas index.

In [ ]:
# Save final dataset
print("\nSaving final processed dataset...")
df.to_excel(output_file, index=False)

## **18. Displaying the Final Processing Summary**
This block prints a general summary of the operation, showing the final shape of the dataset, the paths of the output files, and the complete list of the final columns in order.

In [ ]:
# Final summary
print("\n" + "=" * 50)
print("Processing complete!")
print("=" * 50)
print(f"Final dataset shape: {df.shape}")
print(f"Output file: {output_file}")
print(f"Mappings file: {mappings_file}")
print(f"Total mappings created: {len(all_mappings)}")
print(f"Final columns: {len(df.columns)}")

print("\nFinal columns:")
for i, col in enumerate(df.columns, 1):
  print(f"  {i:2d}. {col}")

## **19. Displaying the Feature Engineering Summary**
The final code block displays a detailed summary of the feature engineering transformations performed, including:

- The temporal features that were created.

- Statistics about the label encoding (how many unique values were mapped).

- A summary of the outlier treatment.

- An analysis of the memory efficiency gained by using label encoding instead of one-hot encoding.

In [ ]:
# Feature engineering summary
print(f"\nFeature engineering summary:")

# Temporal features
temporal_columns = [col for col in df.columns if col.startswith("SERVICE_ORDER_")]
if temporal_columns:
  print(f"Temporal features: {len(temporal_columns)} features from SERVICE ORDER ORIGINAL DATE")
  print(f"  Basic: year, month, day")

# Label encoding
if "MODEL_TYPE_CODE_encoded" in df.columns and original_model_types is not None:
  print(f"MODEL TYPE CODE: {len(original_model_types)} values -> 1 encoded column (0-{len(original_model_types)-1})")
if "PRODUCT_CODE_encoded" in df.columns and original_product_codes is not None:
  print(f"PRODUCT CODE: {len(original_product_codes)} values -> 1 encoded column (0-{len(original_product_codes)-1})")
if "ASSET_CODE_encoded" in df.columns and original_asset_codes is not None:
  print(f"ASSET CODE (vehicles): {len(original_asset_codes)} plates -> 1 encoded column (0-{len(original_asset_codes)-1})")

# Outlier treatment summary
if 'outlier_summary' in locals() and outlier_summary:
  print(f"Outlier treatment:")
  total_outliers = sum(info['outliers_treated'] for info in outlier_summary.values())
  print(f"  Total outliers treated: {total_outliers}")
  for column, info in outlier_summary.items():
    print(f"  {column}: {info['outliers_treated']} outliers ({info['percentage']:.1f}%) capped")

# Memory efficiency
encoded_columns = [col for col in df.columns if col.endswith("_encoded")]
if encoded_columns and (original_model_types is not None or original_product_codes is not None or original_asset_codes is not None):
  potential_ohe_columns = (
    (len(original_model_types) if original_model_types is not None else 0) +
    (len(original_product_codes) if original_product_codes is not None else 0) +
    (len(original_asset_codes) if original_asset_codes is not None else 0)
  )
  print(f"Efficiency: {len(encoded_columns)} label-encoded vs {potential_ohe_columns} potential one-hot columns")
  print(f"Memory savings: ~{potential_ohe_columns - len(encoded_columns)} fewer columns")

### Now the dataset is ready for analysis and model training!